# F4 · Dynamic mode decomposition: learning the modes of a time series, and when they can predict
**Series F: Quantum Meets Machine Learning, Post 4 of 4** &nbsp;|&nbsp; companion *primer* notebook

The full tutorial of this post (*Dynamic Mode Decomposition*, Prof. V. S. Batista, CHEM 584) takes a **movie** of a system, a sequence of snapshots $v_1,\dots,v_n$, and finds from the movie alone a few **modes**, each with its own decay rate and oscillation frequency, and a **prediction** of what happens next. It ends with a puzzle: for an oscillating two-dimensional Gaussian, DMD analyses the movie well but **fails to predict**, and a longer record does not help. This primer rebuilds DMD **from scratch with NumPy**, reproduces the tutorial's numbers, and then answers the puzzle with three experiments: how many frequencies the movie contains, what time-delay embedding does, and what happens when the *wavefunction* (a linear quantity) replaces the Gaussian's modulus.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in well under a minute on a CPU. **Part II** installs the PyDMD package and checks our implementation against it.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | snapshots and the propagator of A → B; eigenmodes; the overlap matrix |
| 2 | continuous time; the SVD and $A=YX^+$ |
| 3 | DMD in six steps, as a function |
| 4 | A → B → C from spectra: choosing $r$, spurious modes, noise, a hold-out test |
| 5 | the moving Gaussian: analysis works, prediction fails |
| 6 | why: spatial rank against temporal frequencies |
| 7 | two repairs: time delays, and the complex wavefunction |
| **Part II** | cross-check with PyDMD |

## 0. Setup

In [ ]:
import math, time, importlib.util
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
rng = np.random.default_rng(0)                          # fixed random numbers: reproducible results

## 1. Snapshots, the propagator, and modes that evolve independently
A first-order reaction A → B; the snapshot is $v=([\mathrm A],[\mathrm B])^T$, and in each time step half of the remaining A turns into B. Starting from pure A, $v_{j+1}=Av_j$ with $A=\begin{pmatrix}1/2&0\\1/2&1\end{pmatrix}$. An eigenvector $\Gamma_k$ is only stretched, $A\Gamma_k=\lambda_k\Gamma_k$, and if $v_1=\sum_kc_k\Gamma_k$ then $v_j=\sum_kc_k\lambda_k^{\,j-1}\Gamma_k$: the coupled problem is a set of independent one-number problems. The eigenvectors of a non-symmetric $A$ are **not orthogonal**, so $c_k=\Gamma_k^\dagger v_1$ is wrong; the correct recipe is $c=S^{-1}\Gamma^\dagger v_1$ with the overlap matrix $S=\Gamma^\dagger\Gamma$.
**Stop and predict:** the eigenvalues of $A$ (one is the final state), and whether $\Gamma^\dagger v_1$ gives the coefficients.

In [ ]:
A_true = np.array([[0.5, 0.0], [0.5, 1.0]]); v1 = np.array([1.0, 0.0]); n = 6
V = np.zeros((2, n)); V[:, 0] = v1
for j in range(1, n): V[:, j] = A_true @ V[:, j - 1]
print("movie V (columns = snapshots), shape", V.shape); print(V)

lam, Gamma = np.linalg.eig(A_true); print("\neigenvalues:", lam, "| residual max|A Gamma - Gamma lambda| =", np.abs(A_true @ Gamma - Gamma * lam).max())
c_naive = Gamma.conj().T @ v1; S = Gamma.conj().T @ Gamma; c = np.linalg.solve(S, Gamma.conj().T @ v1)
print("overlap matrix S (not the identity):\n", S)
print("naive coefficients  ", c_naive, "-> reconstructed v_1 =", Gamma @ c_naive); print("overlap coefficients", c, "-> reconstructed v_1 =", Gamma @ c)
V_modes = (Gamma @ (c[:, None] * lam[:, None] ** np.arange(n)[None, :])).real
assert np.allclose(V_modes, V) and not np.allclose(Gamma @ c_naive, v1)
print("\nthe movie rebuilt from the modes equals the original:", np.allclose(V_modes, V))

## 2. Continuous time, the SVD, and $A=YX^+$
If $\dot v=\mathcal Av$ then one step is $A=e^{\mathcal A\tau}$, with the same eigenvectors and eigenvalues $\lambda_k=e^{\mu_k\tau}$, so $\mu_k=\ln\lambda_k/\tau$: the real part is a growth ($>0$) or decay ($<0$) rate and the imaginary part an angular frequency. With the shifted matrices $X=[v_1\cdots v_{n-1}]$ and $Y=[v_2\cdots v_n]$ we have $Y\approx AX$, and the SVD $X=U\Sigma V^T$ gives the pseudo-inverse $X^+=V\Sigma^{-1}U^T$ and the propagator $A=YX^+$, computed from the data alone.
**Stop and predict:** the decay rate of the mode with $\lambda=\tfrac12$ for $\tau=1$.

In [ ]:
tau = 1.0; mu = np.log(lam.astype(complex)) / tau
print("mu =", mu.real, "| the decaying mode has mu = -ln 2 =", -np.log(2))
X, Y = V[:, :-1], V[:, 1:]; U, s, Vh = np.linalg.svd(X, full_matrices=False)
print("singular values of X:", s, "| shapes U, s, Vh:", U.shape, s.shape, Vh.shape)
A_est = Y @ (Vh.T @ np.diag(1 / s) @ U.T)
print("A estimated from the movie, Y X^+:\n", A_est, "\nmax |A_est - A_true| =", np.abs(A_est - A_true).max())
assert np.allclose(A_est, A_true) and np.isclose(mu.real.min(), -np.log(2))

## 3. DMD in six steps
For a large $N$ the matrix $A=YX^+$ is huge; DMD never builds it. With the truncated SVD ($r$ values kept): **(1)** $X\approx U\Sigma V^T$; **(2)** the small $r\times r$ matrix $\tilde A=U^TYV\Sigma^{-1}$; **(3)** its eigenproblem $\tilde A\tilde\Gamma_k=\lambda_k\tilde\Gamma_k$ (the $\lambda_k$ are eigenvalues of $A$); **(4)** the exact modes $\Gamma_k=\lambda_k^{-1}YV\Sigma^{-1}\tilde\Gamma_k$; **(5)** the amplitudes $c=(\Gamma^\dagger\Gamma)^{-1}\Gamma^\dagger v_1$; **(6)** the prediction $v_j=\sum_kc_k\lambda_k^{\,j-1}\Gamma_k$, which for $j>n$ extrapolates the data.
**Stop and predict:** the shape of $\tilde A$ for $N=5000$ wavelengths, $n=101$ spectra and $r=4$.

In [ ]:
def amplitudes(Gamma, psi0):
    """c = S^-1 Gamma^H psi0 with the overlap matrix S = Gamma^H Gamma (a least-squares fit of psi0 by the modes)."""
    return np.linalg.solve(Gamma.conj().T @ Gamma, Gamma.conj().T @ psi0)

def dmd(V, r=None, eps=1e-10):
    """Exact DMD of a snapshot matrix V (N x n, one snapshot per column); r singular values, or those above eps * sigma_1."""
    X, Y = V[:, :-1], V[:, 1:]
    U, s, Vh = np.linalg.svd(X, full_matrices=False)                              # step 1
    if r is None: r = int(np.sum(s > eps * s[0]))
    Ur, sr, Vr = U[:, :r], s[:r], Vh[:r, :].conj().T
    At = Ur.conj().T @ Y @ Vr @ np.diag(1 / sr)                                   # step 2
    lam, W = np.linalg.eig(At)                                                    # step 3
    Gamma = (Y @ Vr @ np.diag(1 / sr) @ W) / lam[None, :]                         # step 4
    return lam, Gamma, amplitudes(Gamma, V[:, 0]), s                              # step 5

def propagate(lam, Gamma, c, steps):
    """psi_j = sum_k c_k lam_k^j Gamma_k for j = 0 .. steps-1 (columns); steps > n extrapolates."""
    return Gamma @ (c[:, None] * lam[:, None] ** np.arange(steps)[None, :])

lam_f, Gam_f, c_f, s_f = dmd(V, r=2); V_exact = np.array([[0.5 ** j, 1 - 0.5 ** j] for j in range(8)]).T
print("DMD eigenvalues:", np.sort(lam_f.real), "(expected 0.5 and 1)")
print("prediction error for j = 1..8 (6 observed, 2 predicted):", np.abs(propagate(lam_f, Gam_f, c_f, 8).real - V_exact).max())
res = np.abs(A_est @ Gam_f - Gam_f * lam_f).max(); print("the exact modes are eigenvectors of A = Y X^+: residual", f"{res:.1e}")
assert np.abs(propagate(lam_f, Gam_f, c_f, 8).real - V_exact).max() < 1e-10 and res < 1e-10

## 4. A → B → C from time-resolved spectra
Consecutive reactions A → B → C with $k_1=0.30$ and $k_2=0.10$; a spectrometer records $N=300$ absorbances at $n=40$ times, with noise $0.002$; each snapshot is $v(t)=[\mathrm A]\varepsilon_A+[\mathrm B]\varepsilon_B+[\mathrm C]\varepsilon_C$. The dynamics is linear, so DMD should find $\lambda=e^{-k_1\tau},e^{-k_2\tau},1$ and predict the next 20 spectra. Choose $r$ from the **gap** in the singular values.
**Stop and predict:** how many singular values are above the noise, and what happens with $r=2$?

In [ ]:
k1, k2, tau, n_all = 0.30, 0.10, 1.0, 60
def make_kinetics(k1, k2, noise, rng):
    t = np.arange(n_all) * tau; cA = np.exp(-k1 * t); cB = k1 / (k2 - k1) * (np.exp(-k1 * t) - np.exp(-k2 * t)); conc = np.vstack([cA, cB, 1 - cA - cB])
    wl = np.linspace(400, 700, 300); band = lambda c0, w: np.exp(-0.5 * ((wl - c0) / w) ** 2)
    eps_spec = np.vstack([1.0 * band(480, 25), 0.7 * band(540, 30), 1.2 * band(610, 20)]).T
    Vc = eps_spec @ conc; return Vc + noise * rng.normal(size=Vc.shape), Vc
V_all, V_clean = make_kinetics(k1, k2, 0.002, np.random.default_rng(0)); n_obs = 40; V_obs = V_all[:, :n_obs]
s_obs = np.linalg.svd(V_obs[:, :-1], compute_uv=False); print("sigma_k / sigma_1, k = 1..6:", (s_obs[:6] / s_obs[0]).round(4), "-> a gap after 3")
relerr = lambda R, T: np.linalg.norm(R - T, axis=0) / np.linalg.norm(T, axis=0)
rows = []
for r in (1, 2, 3, 5, 10, 20):
    lam, G, c, _ = dmd(V_obs, r=r); R = propagate(lam, G, c, n_all).real; e = relerr(R, V_clean)
    imp = np.abs(c) * np.linalg.norm(G, axis=0); top = np.argsort(-imp)[:3]; rate = -np.log(lam.astype(complex)).real / tau
    rows.append((r, e[:n_obs].mean(), e[n_obs:].mean(), np.sort(rate[top])[::-1], rate.max()))
print(pd.DataFrame([(r, a, b, str(np.round(t3, 4)), f) for r, a, b, t3, f in rows], columns=["r", "window error", "prediction error", "decay rates of the 3 largest modes", "fastest rate in the list"]).round(4).to_string(index=False))
r3 = rows[2]; assert r3[1] < 0.003 and r3[2] < 0.003 and abs(r3[3][0] - 0.3) < 0.01 and abs(r3[3][1] - 0.1) < 0.005 and rows[1][2] > 0.05 and rows[0][2] > 0.5
print("\nr = 3 recovers the rate constants; r = 2 loses a process; for r >= 5 the error is unchanged but the list contains FAST spurious modes (rates far above 0.3) with tiny amplitudes: never read them as processes")

**Noise, close rates, and a hold-out test.** (i) Raise the noise to $0.02$. (ii) Move $k_2$ towards $k_1=0.3$: two similar rates are hard to tell apart. (iii) Choose $r$ without looking at the singular values: **hold out the last 5 observed snapshots**, fit on the rest, and plot the prediction error against $r$; the right $r$ is where the curve stops improving.

In [ ]:
def rates_of(V, r, top=3):
    lam, G, c, _ = dmd(V, r=r); imp = np.abs(c) * np.linalg.norm(G, axis=0); idx = np.argsort(-imp)[:top]; return np.sort((-np.log(lam.astype(complex)).real / tau)[idx])[::-1]
rows = []
for noise, kk2 in ((0.002, 0.10), (0.02, 0.10), (0.002, 0.20), (0.002, 0.25), (0.02, 0.25)):
    Va, Vc = make_kinetics(k1, kk2, noise, np.random.default_rng(0)); rr = rates_of(Va[:, :n_obs], 3); true = np.array([k1, kk2, 0.0])
    rows.append((noise, kk2, str(rr.round(4)), np.abs(rr - np.sort(true)[::-1]).max()))
print(pd.DataFrame(rows, columns=["noise", "k2", "recovered rates", "largest error"]).round(4).to_string(index=False))

def holdout_error(V, r, h=5):
    lam, G, c, _ = dmd(V[:, :V.shape[1] - h], r=r); R = propagate(lam, G, c, V.shape[1]); return relerr(R[:, -h:], V[:, -h:]).mean()
ho = {r: holdout_error(V_obs, r) for r in (1, 2, 3, 4, 5, 8, 12)}; print("\nhold-out prediction error by r (kinetics):", {r: round(float(v), 4) for r, v in ho.items()})
assert ho[2] > 5 * ho[3] and abs(ho[8] - ho[3]) < 0.5 * ho[3]
print("the curve falls steeply up to r = 3 and is flat afterwards: r = 3 is the smallest sufficient rank")

## 5. A moving Gaussian: analysis works, prediction fails
$\psi(x,y,t)=e^{-(x-x_e)^2-(y-y_e)^2}$ with $x_e=A_x\cos\omega_xt$, $y_e=A_y\cos\omega_yt$, $A_x=0.5$, $A_y=0.1$, $\omega_x=\pi/30$, $\omega_y=0.5+\pi/30$, on a $50\times50$ grid with $-2\le x,y<2$; $n=30$ snapshots at $t=0,1,\dots,29$, and 30 more kept to test the prediction. Each snapshot has $N=2500$ numbers.
**Stop and predict:** is a prediction from 30 snapshots (half a period of the slow oscillation) going to be reliable?

In [ ]:
npt, L, nt, n_extra = 50, 4.0, 30, 30; Ax, Ay = 0.5, 0.1; wx, wy = np.pi / 30, 0.5 + np.pi / 30
z = (np.arange(npt) - npt / 2) * (L / npt); N = npt * npt
def real_movie(n_steps):
    k = np.arange(n_steps); gx = np.exp(-(z[:, None] - (Ax * np.cos(wx * k))[None, :]) ** 2); gy = np.exp(-(z[:, None] - (Ay * np.cos(wy * k))[None, :]) ** 2)
    return np.einsum("ik,jk->ijk", gx, gy).reshape(N, n_steps)
Vg_all = real_movie(nt + n_extra); Vg = Vg_all[:, :nt]
sg = np.linalg.svd(Vg[:, :-1], compute_uv=False); print("snapshot matrix", Vg.shape, "| singular values above 1e-5 sigma_1:", int(np.sum(sg > 1e-5 * sg[0])))
lam_g, Gam_g, c_g, _ = dmd(Vg, eps=1e-5); order = np.argsort(-np.abs(c_g) * np.linalg.norm(Gam_g, axis=0)); lam_g, Gam_g, c_g = lam_g[order], Gam_g[:, order], c_g[order]
e_g = relerr(propagate(lam_g, Gam_g, c_g, nt + n_extra).real, Vg_all)
print(f"r = {len(lam_g)}, max|lambda| = {abs(lam_g).max():.4f} | error in the observed window {e_g[:nt].mean():.3f}, in the next 30 steps {e_g[nt:].mean():.3f} (max {e_g[nt:].max():.3f})")
imp = np.abs(c_g) * np.linalg.norm(Gam_g, axis=0)
print(f"known omega_x tau = {wx:.4f}, omega_y tau = {wy:.4f}\n rank  |lambda|  arg(lambda)  relative amplitude")
for k in range(6): print(f" {k + 1:4d}  {abs(lam_g[k]):.4f}   {np.angle(lam_g[k]):+.4f}      {imp[k] / imp[0]:.3f}")
print("\neps scan (observed window / next 30 steps / largest |lambda|):")
for ee in (1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8):
    l, G2, c2, _ = dmd(Vg, eps=ee); e2 = relerr(propagate(l, G2, c2, nt + n_extra).real, Vg_all); print(f"  eps {ee:.0e}: r = {len(l):2d}   {e2[:nt].mean():.3f}   {e2[nt:].mean():.3f}   {abs(l).max():.3f}")
assert len(lam_g) == 17 and abs(e_g[:nt].mean() - 0.033) < 0.005 and abs(e_g[nt:].mean() - 0.31) < 0.03 and abs(abs(np.angle(lam_g[1])) - 0.1023) < 0.002
print("\nthe slow frequency is found (0.102 against 0.105); the movie is compressed to 17 modes; the prediction is wrong by about 30 %")

In [ ]:
data_long = real_movie(150); rows = []
for ntr in (30, 60, 90, 120):
    l, G2, c2, _ = dmd(data_long[:, :ntr], eps=1e-5); e2 = relerr(propagate(l, G2, c2, ntr + 30).real, data_long[:, :ntr + 30]); rows.append((ntr, len(l), e2[:ntr].mean(), e2[ntr:].mean()))
print(pd.DataFrame(rows, columns=["snapshots observed", "r", "error in the window", "error in the next 30"]).round(3).to_string(index=False))
holds = {r: holdout_error(Vg, r) for r in (2, 3, 5, 8, 11, 17)}; print("\nhold-out prediction error by r (movie):", {r: round(float(v), 3) for r, v in holds.items()})
print("the best hold-out error is", round(min(holds.values()), 2), "against", round(e_g[:nt].mean(), 3), "inside the window: the gap warns against extrapolating, before any extrapolation is made")
assert min(holds.values()) > 0.1 and rows[-1][3] > 0.2

## 6. Why: spatial rank against temporal frequencies
The movie lives in a 17-dimensional space (17 singular values above $10^{-5}\sigma_1$, for any record length). A DMD model with $r$ modes has $r$ eigenvalues, hence at most $r$ frequencies (a conjugate pair counts twice). But the amplitude of each spatial pattern is a *polynomial in $\cos\omega_xt$ and $\cos\omega_yt$*: it contains the combinations $k\omega_x\pm l\omega_y$, many more than 17 frequencies. The number of distinct exponentials in a time series is the numerical rank of its **Hankel matrix** (the matrix of its overlapping windows), which we can compute.
**Stop and predict:** is the Hankel rank of the leading coefficient below or above 17?

In [ ]:
Vlong = real_movie(400); U, s, Vh = np.linalg.svd(Vlong, full_matrices=False); coef = Vh[:17] * s[:17, None]       # time series of the 17 spatial patterns
print("singular values above 1e-5 sigma_1 on 400 snapshots:", int(np.sum(s > 1e-5 * s[0])))
def hankel_rank(a, d=150, tol=1e-5):
    H = np.array([a[i:i + len(a) - d + 1] for i in range(d)]); sv = np.linalg.svd(H, compute_uv=False); return int(np.sum(sv > tol * sv[0]))
hr = [hankel_rank(coef[k]) for k in range(17)]
print("number of distinct exponentials (Hankel rank, tolerance 1e-5) in the time series of patterns 1, 5, 9, 13, 17:", [hr[k] for k in (0, 4, 8, 12, 16)])
print("a model with at most 17 modes cannot represent a time series with", max(hr), "frequencies")
assert hr[0] < 20 and max(hr) > 60

## 7. Two repairs
**(a) Time delays.** Stack $d$ consecutive snapshots into one tall state $(v_j,v_{j+1},\dots,v_{j+d-1})$ and apply DMD to the stacked movie (a *Hankel* or delay-embedded DMD). The state is now $dN$-dimensional, so the model can have up to $dr$ modes, enough for many frequencies. It needs a record long enough to determine them.
**(b) The linear quantity.** The Gaussian $e^{-(x-x_e)^2}$ is the **modulus** of the wavefunction of a *coherent state* of a harmonic oscillator ($\hbar=1$, $m\omega=2$ in each direction): $\psi_x=\exp\{-(x-x_c)^2+i[p_cx-p_cx_c/2-\omega t/2]\}$ with $x_c=A\cos\omega t$ and $p_c=-m\omega A\sin\omega t$. The modulus is a nonlinear function of the state; the **complex wavefunction obeys the linear Schrödinger equation**, $\psi_{j+1}=e^{-iH\tau}\psi_j$, and DMD on it should recover the propagator, whose eigenphases are $-E_n\tau$.
**Stop and predict:** with 30 snapshots, will (a) help? Will (b)?

In [ ]:
def hankel_dmd(Vobs, d, eps, n_pred):
    nn = Vobs.shape[1]; Hm = np.vstack([Vobs[:, i:nn - d + 1 + i] for i in range(d)]); lam, G, c, _ = dmd(Hm, eps=eps)
    return lam, propagate(lam, G, c, nn - d + 1 + n_pred)[:N].real
rows = []; Vall = real_movie(150)
for nobs in (30, 120):
    for d in (1, 2, 4, 8):
        lam, R = hankel_dmd(Vall[:, :nobs], d, 1e-5, 30 + d - 1); e = relerr(R, Vall[:, :R.shape[1]])
        rows.append((nobs, d, len(lam), e[:nobs - d + 1].mean(), e[nobs - d + 1:nobs - d + 31].mean()))
print(pd.DataFrame(rows, columns=["snapshots", "delays d", "modes", "window error", "error in the next 30"]).round(4).to_string(index=False))
tab = {(a, b): p for a, b, m, w, p in rows}; assert tab[(30, 4)] > 0.1 and tab[(120, 4)] < 0.01 and tab[(120, 1)] > 0.2
print("\nwith 30 snapshots delays do not help (half a period); with 120 snapshots (two periods) 4 delays predict to", f"{tab[(120, 4)]:.4f}")

In [ ]:
def coherent(n_steps, mw=2.0):
    """Complex wavefunction of a 2D coherent state (hbar = 1, m*omega = 2 per direction): |psi| equals the real movie."""
    k = np.arange(n_steps).astype(float)
    def one(A, w):
        xc = A * np.cos(w * k); pc = -mw * A * np.sin(w * k)
        return np.exp(-(z[:, None] - xc[None, :]) ** 2 + 1j * (pc[None, :] * z[:, None] - (pc * xc / 2)[None, :] - (w * k / 2)[None, :]))
    return np.einsum("ik,jk->ijk", one(Ax, wx), one(Ay, wy)).reshape(N, n_steps)
Cx = coherent(150); assert np.allclose(np.abs(Cx[:, :60]), real_movie(60))
print("|psi| of the complex movie equals the real movie of Section 5: True")
rows = []
for ee in (1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8):
    l, G2, c2, _ = dmd(Cx[:, :30], eps=ee); e2 = relerr(propagate(l, G2, c2, 60), Cx[:, :60]); rows.append((f"{ee:.0e}", len(l), e2[:30].mean(), e2[30:].mean(), abs(l).max()))
print(pd.DataFrame(rows, columns=["eps", "modes", "window error", "error in the next 30", "max |lambda|"]).round(5).to_string(index=False))
for nobs in (60, 90):
    l, G2, c2, _ = dmd(Cx[:, :nobs], eps=1e-6); e2 = relerr(propagate(l, G2, c2, nobs + 30), Cx[:, :nobs + 30]); print(f"observed {nobs}: {len(l)} modes, error in the next 30: {e2[nobs:].mean():.5f}")
l30, G30, c30, _ = dmd(Cx[:, :30], eps=1e-6); e30 = relerr(propagate(l30, G30, c30, 60), Cx[:, :60])
assert e30[30:].mean() < 0.03 and abs(l30).max() < 1.001 and e_g[nt:].mean() > 10 * e30[30:].mean()
print("\nthe same 30 snapshots, as a complex wavefunction: the prediction error falls from 31 % to about 1 %, with all |lambda| on the unit circle (unitary dynamics)")

**The eigenphases are energy levels.** The wavefunction is a superposition of oscillator eigenstates, $\psi=\sum c_n\phi_n e^{-iE_nt}$ with $E_{n_xn_y}=(n_x+\tfrac12)\omega_x+(n_y+\tfrac12)\omega_y$, and the coherent-state amplitudes are $c_n/c_0=\alpha^n/\sqrt{n!}$ with $\alpha_x=A_x\sqrt{m\omega/2}=0.5$ and $\alpha_y=0.1$. DMD should find them: eigenphase $=-E_n\tau$ and relative amplitude $=\alpha_x^{n_x}\alpha_y^{n_y}/\sqrt{n_x!\,n_y!}$.

In [ ]:
imp = np.abs(c30) * np.linalg.norm(G30, axis=0); order = np.argsort(-imp); l30, imp = l30[order], imp[order] / imp[order][0]
wrap = lambda a: (a + np.pi) % (2 * np.pi) - np.pi
levels = {(a, b): -((a + .5) * wx + (b + .5) * wy) for a in range(6) for b in range(3)}; alpha = {0: Ax, 1: Ay}
rows = []
for k in range(6):
    ph = np.angle(l30[k]); q = min(levels, key=lambda q: abs(wrap(levels[q] - ph))); amp = Ax ** q[0] * Ay ** q[1] / math.sqrt(math.factorial(q[0]) * math.factorial(q[1]))
    rows.append((k + 1, abs(l30[k]), ph, q, levels[q], wrap(ph - levels[q]), imp[k], amp))
print(pd.DataFrame(rows, columns=["rank", "|lambda|", "eigenphase", "(nx, ny)", "-E tau", "difference", "DMD amplitude", "alpha^n / sqrt(n!)"]).round(4).to_string(index=False))
assert all(abs(r[5]) < 0.003 for r in rows) and all(abs(r[6] - r[7]) < 0.01 for r in rows) and [r[3] for r in rows[:3]] == [(0, 0), (1, 0), (2, 0)]
print("\nthe zero-point energy, the oscillator spacings and the Poisson amplitudes of the coherent state are recovered from 30 snapshots")

In [ ]:
th = np.arange(60)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.semilogy(th, e_g[:60] + 1e-6, "o-", ms=3, color=COLORS["red"], label="modulus (real movie), plain DMD, 17 modes")
ax.semilogy(th, e30 + 1e-6, "s-", ms=3, color=COLORS["blue"], label="complex wavefunction, DMD, 17 modes")
ax.axvline(nt - 0.5, color="k", ls="--"); ax.set(xlabel="time step", ylabel="relative error", title="The same 30 snapshots: right of the line is prediction"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Your turn
**Exercise 1.** In Section 4 set the noise to $0.1$. Which $r$ does the hold-out test choose, and are the rate constants still recovered?
**Exercise 2.** Add a fourth species (C → D, $k_3=0.05$). How many singular values are above the noise, and what rates does DMD return?
**Exercise 3.** Aliasing: set $\tau=6$ for a signal with $\omega=1$ (so $\omega\tau=6>\pi$) and compare $\arg\lambda/\tau$ with $\omega$.
**Exercise 4.** In Section 5 estimate the amplitudes by least squares over **all** snapshots instead of the first one. What happens to the window error and the prediction?
**Exercise 5.** In Section 7 use $A_x=0.7$, $A_y=0.3$: how do the Poisson amplitudes and the number of modes change?

## 9. Self-check

In [ ]:
assert np.allclose(A_est, A_true) and np.abs(propagate(lam_f, Gam_f, c_f, 8).real - V_exact).max() < 1e-10
assert r3[1] < 0.003 and ho[2] > 5 * ho[3] and len(lam_g) == 17 and abs(e_g[nt:].mean() - 0.31) < 0.03
assert max(hr) > 60 and tab[(120, 4)] < 0.01 and e30[30:].mean() < 0.03
print("All checks passed: you reproduced the numbers in the post.")

# Part II. Cross-check with PyDMD
> **Read this first.** Part II installs **PyDMD** (`pip install pydmd`), a widely used DMD package, and checks the implementation of Part I against it on the spectra of Section 4: the eigenvalues, the modes (up to a scale factor and sign) and the reconstruction. PyDMD takes the snapshots in the **columns**, as we do; its eigenvalues and exact modes follow the same six steps, so agreement to rounding error is expected.

In [ ]:
!pip install -q pydmd

In [ ]:
HAVE_PYDMD = importlib.util.find_spec("pydmd") is not None
if HAVE_PYDMD:
    from pydmd import DMD
    ours_l, ours_G, ours_c, _ = dmd(V_obs, r=3)
    try:
        dm = DMD(svd_rank=3, exact=True); dm.fit(V_obs)
        l_py = np.asarray(dm.eigs); order_o = np.argsort(-ours_l.real); order_p = np.argsort(-l_py.real)
        print("eigenvalues  ours  :", ours_l[order_o].round(6)); print("eigenvalues  PyDMD :", l_py[order_p].round(6))
        print("max |difference| of the eigenvalues:", f"{np.abs(ours_l[order_o] - l_py[order_p]).max():.1e}")
        M_py = np.asarray(dm.modes)[:, order_p]; M_o = ours_G[:, order_o]
        cos = [abs(np.vdot(M_o[:, k], M_py[:, k])) / (np.linalg.norm(M_o[:, k]) * np.linalg.norm(M_py[:, k])) for k in range(3)]
        print("alignment of the modes (1 = same up to scale and sign):", np.round(cos, 8))
        R_py = np.asarray(dm.reconstructed_data).real; R_o = propagate(ours_l, ours_G, ours_c, V_obs.shape[1]).real
        print("max |reconstruction ours - PyDMD| (observed window):", f"{np.abs(R_py[:, :V_obs.shape[1]] - R_o).max():.1e}")
        assert np.abs(ours_l[order_o] - l_py[order_p]).max() < 1e-8
    except Exception as e:
        print("the comparison could not be completed with this PyDMD version:", type(e).__name__, e)
else:
    print("PyDMD is not available: run the install cell above (restart the session if Colab asks), then re-run this cell.")

### What to look for
* **Eigenvalues** agree to rounding error (about $10^{-12}$ to $10^{-8}$): both implementations diagonalize the same $r\times r$ matrix $\tilde A=U^TYV\Sigma^{-1}$.
* **Modes** have an alignment of $1$: they are the same vectors up to a complex factor.
* **Reconstruction** agrees as well, provided both fit the amplitudes by least squares from the first snapshot (the default of PyDMD with `exact=True`); a difference points to a different amplitude convention, not to a different model.
* Try `DMD(svd_rank=3, tlsq_rank=3)` (total-least-squares DMD, which corrects the bias that noise causes in the eigenvalues) and compare the eigenvalues with the noise raised to $0.02$.